# Bits e bytes em C

*Rode uma célula com ▶ (ou Shift+Enter): o Sphinx compila e roda a célula no seu computador. Mude o código e rode de novo: este notebook é a sua cópia.*

**Depois desta unidade você consegue:**
- Ler, ligar, desligar e inverter bits com &, |, ^, ~ e deslocamentos, em tipos sem sinal de largura fixa como uint32_t.
- Empacotar vários campos pequenos num inteiro e desempacotá-los de novo com máscaras e deslocamentos.
- Olhar os bytes e bits de qualquer valor, como um float, copiando-o para um inteiro com memcpy.

Os **exercícios** deste notebook terminam com linhas `check(...)`: testes mínimos que imprimem ✓ quando o seu código está certo. Rode a próxima célula uma vez antes, para eles poderem usar o `check`.

In [ ]:
// shared: check_int e check_str imprimem ✓ ou ✗; os exercícios abaixo usam
void check_int(const char *what, long actual, long expected) {
    if (actual == expected) {
        printf("✓ %s\n", what);
    } else {
        printf("✗ %s: esperado %ld, veio %ld\n", what, expected, actual);
    }
}

void check_str(const char *what, const char *actual, const char *expected) {
    if (actual == NULL) {
        actual = "(NULL)";
    }
    if (strcmp(actual, expected) == 0) {
        printf("✓ %s\n", what);
    } else {
        printf("✗ %s: esperado \"%s\", veio \"%s\"\n", what, expected, actual);
    }
}

## Em resumo

O C trabalha direto nos bits dos inteiros: `&` (e), `|` (ou), `^` (ou exclusivo), `~` (não), `<<` e `>>` (deslocamentos). Use tipos **sem sinal** de largura fixa de `<stdint.h>`, como `uint32_t`: o tamanho deles é certo, e os deslocamentos são totalmente definidos.

```c
#include <stdio.h>
#include <stdint.h>
#include <string.h>

int main(void) {
    uint32_t flags = 0;
    flags |= 1u << 3;                     // liga o bit 3
    flags ^= 1u << 0;                     // inverte o bit 0
    int bit3 = (flags >> 3) & 1;          // lê o bit 3
    flags &= ~(1u << 0);                  // desliga o bit 0
    printf("%u %d\n", (unsigned) flags, bit3);   // 8 1

float f = 1.0f;
    uint32_t bits;
    memcpy(&bits, &f, sizeof bits);       // os próprios bits do float
    printf("0x%08X\n", (unsigned) bits); // 0x3F800000
    return 0;
}
```

Uma **máscara** seleciona um campo: `(value >> 5) & 0x3F` pega os 6 bits que começam no bit 5.

#### 🎞️ Lendo um campo de bits com shift e máscara

![Para ler os bits 5 a 10 de 0xABCD, desloque 5 para a direita para trazê-los para baixo e faça AND com 0x3F para ficar com 6 bits: 011110, ou 30](shift-mask-steps.pt-br.svg)

**Deslocamentos, máscaras e o "dar a volta":** mude `value` e `start` para ler outros campos.

In [ ]:
uint16_t value = 0xABCD;
int start = 5;
unsigned field = (value >> start) & 0x3F;         /* 6 bits a partir do bit 5 */
printf("0x%04X: os bits %d..%d valem %u (0x%02X)\n", (unsigned) value, start, start + 5, field, field);
for (int k = 0; k < 8; k++) {
    printf("1u << %d = %3u\n", k, 1u << k);
}
uint8_t byte = 255;
byte = (uint8_t) (byte + 1);                      /* dá a volta para 0 */
printf("255 + 1 num uint8_t: %u\n", (unsigned) byte);

### ✏️ Exercício: Ligue um bit

Devolva x com o bit k em 1, usando `|` e um 1 deslocado. Um bit que já está ligado continua ligado.

In [ ]:
unsigned set_bit(unsigned x, int k) {
    // TODO: faça OR de x com 1u << k
    return x;
}

int main(void) {
    check_int("bit 3 de 0", set_bit(0, 3), 8);
    check_int("já ligado", set_bit(8, 3), 8);
    check_int("bit 4 de 1", set_bit(1, 4), 17);
    return 0;
}

<details>
<summary>Ver uma resposta</summary>

```c
unsigned set_bit(unsigned x, int k) {
    return x | (1u << k);
}

int main(void) {
    check_int("bit 3 de 0", set_bit(0, 3), 8);
    check_int("já ligado", set_bit(8, 3), 8);
    check_int("bit 4 de 1", set_bit(1, 4), 17);
    return 0;
}
```

</details>

**Cuidado:** desloque só valores sem sinal, por menos que a largura deles: os outros deslocamentos são comportamento indefinido ou dependem do compilador.

### ✏️ Exercício: O nibble de baixo

Devolva os 4 bits mais baixos de x (um dígito hexa) com uma máscara.

In [ ]:
unsigned low_nibble(unsigned x) {
    // TODO: faça AND de x com 0xF
    return 0;
}

int main(void) {
    check_int("0xAB", low_nibble(0xAB), 0xB);
    check_int("0x30", low_nibble(0x30), 0);
    check_int("0xFFFF", low_nibble(0xFFFF), 15);
    return 0;
}

<details>
<summary>Ver uma resposta</summary>

```c
unsigned low_nibble(unsigned x) {
    return x & 0xFu;
}

int main(void) {
    check_int("0xAB", low_nibble(0xAB), 0xB);
    check_int("0x30", low_nibble(0x30), 0);
    check_int("0xFFFF", low_nibble(0xFFFF), 15);
    return 0;
}
```

</details>

## Leituras

- [Bitwise Operations](https://beej.us/guide/bgc/html/split/bitwise-operations.html) (Beej's Guide to C Programming, chapter 24, 4 min): AND, OR, XOR e NOT, depois os deslocamentos, e o aviso sobre deslocar valores com sinal. Curto: os operadores são os mesmos que você usou em Fundamentos da Computação, agora em C.
- [Fixed Width Integer Types](https://beej.us/guide/bgc/html/split/fixed-width-integer-types.html) (Beej's Guide to C Programming, chapter 37, 5 min): Opcional: uint8_t, uint32_t e parentes, os limites deles, e os especificadores de formato no estilo PRIu32. Os compiladores do Sphinx também imprimem uint32_t com %u, que os desafios usam.

## Pratique

O notebook cobriu o básico. Agora aplique nos desafios da unidade, na barra lateral do Sphinx, onde o Submit roda todos os testes, inclusive os ocultos:

- **Contar os bits** · Fácil
- **Cores RGB565** · Fácil
- **Anatomia de um float** · Médio
- **CRC-32** · Difícil